In [ ]:
import pandas as pd
import os
import pyodbc as py
from sqlalchemy import create_engine
import urllib
import re
import ftfy

### 1. Camada de ingestão de dados

Estabelece conexão com o banco de dados local utilizando a biblioteca SQLalchemy. Essa etapa cria a estrutura de conexão necessária para que o Pandas extraia os dados brutos diretamente da fonte, assim iniciando o processo ETL.

In [ ]:
driver = 'ODBC Driver 17 for SQL Server'
server = r'localhost\SQLEXPRESS'
database = 'Contoso V2 10M'

params = urllib.parse.quote_plus(
    f'DRIVER={driver};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
    f'CHARSET=UTF8;'
)
connection_string = f"mssql+pyodbc:///?odbc_connect={params}"
engine = create_engine(connection_string)

### 2. Camada de extração e transformação
Nessa etapa os dados são coletados do SQL Server, padronizados e otimizados para uma redução de peso nos arquivos (CSV) finais.

In [ ]:
"""
    Célula que coleta os dados gerais de venda, removendo apenas as lojas digitais. Pegando pedaços (Chunk) por vez para evitar estouro de memória. Dentro destes pedaços são feitos "downcast" para uma melhor eficiência de tipagem nas colunas, também é feita a conversão dos valores para dólar (USD), cálculo de receita (Revenue) e lucro (Profit), além disso foi feito um arredondamento para duas casas decimais após a vírgula.
"""
Query = """
   SELECT Sales.StoreKey,
           Store.Name,
           Store.State,
           Store.Country,
           Sales.CustomerKey,
           Sales.ProductKey,
           Sales.Quantity,
           Sales.[Unit Price],
           Sales.[Net Price],
           Sales.[Unit Cost],
           Sales.[Currency Code],
           Sales.[Exchange Rate],
           Sales.[Order Date]
           FROM Sales
           LEFT JOIN Store ON Sales.StoreKey = Store.StoreKey
           WHERE Sales.StoreKey <> 999999;
           """
try:
    chunk_size = 1000000
    chunk_list = []

    for c in pd.read_sql(Query, engine, chunksize=chunk_size):
        c['StoreKey'] = pd.to_numeric(c['StoreKey'], downcast='integer')
        c['CustomerKey'] = pd.to_numeric(c['CustomerKey'], downcast='integer')
        c['ProductKey'] = pd.to_numeric(c['ProductKey'], downcast='integer')
        c['Quantity'] = pd.to_numeric(c['Quantity'], downcast='integer')

        mask = c['Currency Code'] != 'USD'
        c.loc[mask, 'Unit Price'] = c['Unit Price'] * c['Exchange Rate']
        c.loc[mask, 'Net Price'] = c['Net Price'] * c['Exchange Rate']
        c.loc[mask, 'Unit Cost'] = c['Unit Cost'] * c['Exchange Rate']

        c['StoreKey'] = pd.to_numeric(c['StoreKey'], downcast='integer')
        c['ProductKey'] = pd.to_numeric(c['ProductKey'], downcast='integer')
        c['Quantity'] = pd.to_numeric(c['Quantity'], downcast='integer')

        c['Name'] = c['Name'].astype('category')
        c['State'] = c['State'].astype('category')
        c['Country'] = c['Country'].astype('category')

        c['Order Date'] = pd.to_datetime(c['Order Date'])

        c = c.rename(columns={
            'Unit Price': 'Unit Price USD',
            'Net Price': 'Net Price USD',
            'Unit Cost': 'Unit Cost USD'
            })
        c['Revenue'] = c['Net Price USD'] * c['Quantity']
        c['Profit'] = (c['Net Price USD'] - c['Unit Cost USD']) * c['Quantity']

        c = c.drop(columns=['Exchange Rate', 'Currency Code'])
        chunk_list.append(c)

    sales = pd.concat(chunk_list, ignore_index=True)
    del chunk_list

    sales['Unit Price USD'] = sales['Unit Price USD'].round(2)
    sales['Net Price USD'] = sales['Net Price USD'].round(2)
    sales['Unit Cost USD'] = sales['Unit Cost USD'].round(2)
    sales['Revenue'] = sales['Revenue'].round(2)
    sales['Profit'] = sales['Profit'].round(2)
    print('Connected')

except py.Error as error:
    print(f'Conection error {error.args[0]}')

In [ ]:
"""
    Aqui é coletada a tabela com as informações dos clientes. Como a tabela é menor, não foi utilizada a coleta em pedaços.
"""

Query = """
    SELECT
        CustomerKey, Name, Age, Gender,Country
        FROM Customer;
        """
try:
    customers = pd.read_sql(Query, engine)

    customers['Name'] = customers['Name'].astype('category')
    customers['Gender'] = customers['Gender'].astype('category')
    customers['Country'] = customers['Country'].astype('category')

    customers['Age'] = pd.to_numeric(customers['Age'], downcast='integer')
    customers['CustomerKey'] = pd.to_numeric(
        customers['CustomerKey'],
        downcast='integer')

    print('Connected')

except py.Error as error:
    print(f'Connection error {error.args[0]}')

In [ ]:
"""
    Ao analisar os dados, foi notado que alguns nomes de clientes continham números e espaços em branco em seu início. Foi feita uma função usando a biblioteca Regex para a remoção dos espaços vazios e dos números, e ftfy para a correção de erros mojibake. No fim, aplicando a função nas categorias utilizadas pela coluna de nomes.
"""
def corrected_names(name):
    if pd.isna(name):
        return name
    name = re.sub(r'^\s*[0-9]+', '', str(name)).strip()

    try:
        return ftfy.fix_text(name)

    except UnicodeError:
        return name


names = customers['Name'].cat.categories.map(corrected_names)
customers['Name'] = customers['Name'].cat.rename_categories(names)

In [ ]:
"""
    Já esta célula coleta as informações dos produtos. Como na coluna nome do produto (Product name), foi reparado que no início de cada nome havia o nome da empresa que o criou e no fim de cada nome havia a sua cor, como ambas as informações estão em colunas separadas, esses dados foram removidos para que fique apenas o nome na coluna.
"""

Query = """
    SELECT
        ProductKey, [Product Name], Manufacturer, Color, Category
        FROM Product;
        """
try:
    products = pd.read_sql(Query, engine)

    products['Product Name'] = products['Product Name'].str.split(' ').apply(
        lambda x: ' '.join(x[1:-1])
    ).astype('category')
    products['Product Name'] = products['Product Name'].astype('category')
    products['Color'] = products['Color'].astype('category')
    products['Category'] = products['Category'].astype('category')

    products['ProductKey'] = pd.to_numeric(
        products['ProductKey'],
        downcast='integer')

    print('Connected')

except py.Error as error:
    print(f'Conection error {error.args[0]}')

In [ ]:
"""
    Cria um DataFrame apenas com as informações da loja, também é feito o cálculo do item mais vendido de cada uma.
"""

stores = sales[['StoreKey', 'Name']].drop_duplicates()

most_sold = (
    sales
    .groupby(['StoreKey', 'ProductKey'])
    .size()
    .reset_index(name='Amount')
)
most_sold = (
    most_sold
    .merge(
        products[['ProductKey', 'Product Name']],
        on='ProductKey',
        how='left')
)
most_sold_product = (
    most_sold
    .sort_values(['StoreKey', 'Amount'], ascending=False)
    .drop_duplicates('StoreKey', keep='first')
)
stores = (
    stores
    .merge(most_sold_product,
           on='StoreKey',
           how='left')
)
stores = stores.rename(columns={'ProductKey': 'Best-Selling Unit'})

### 3. Camada de Carga
Etapa final onde os DataFrames processados são exportados para arquivos CSV, servindo como base de dados limpa e otimizada para ser utilizada no Power BI.

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
d = r'.'
sales.to_csv(os.path.join(d, 'sales.csv'), index=False)
products.to_csv(os.path.join(d, 'products.csv'), index=False)
customers.to_csv(os.path.join(d, 'customers.csv'),
                 index=False,
                 encoding="utf-8-sig"
                )
stores.to_csv(os.path.join(d, 'stores.csv'), index=False)